# Artwork OpenAI Pricer

Price artwork auction lots with GPT-4.1 Mini. Input paths are defined explicitly below so the loaded files are visible before running API calls.


## Configuration


In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "artwork_prompt_builder.py").exists():
    ARTWORK_DIR = NOTEBOOK_DIR
else:
    ARTWORK_DIR = Path("Base_OpenAI/artwork").resolve()

BASE_OPENAI = ARTWORK_DIR.parent
WORKSPACE_ROOT = BASE_OPENAI.parent
sys.path.insert(0, str(BASE_OPENAI))
sys.path.insert(0, str(ARTWORK_DIR))

OUTPUT_DIR = ARTWORK_DIR / "outputs"
ARTWORK_DATA_DIR = WORKSPACE_ROOT / "Data" / "sampled_sets_100k"
QUARTERS = ("2025Q1", "2025Q2", "2025Q3", "2025Q4")
QUARTER_FILES = {quarter: ARTWORK_DATA_DIR / f"Artwork_2026_finetune_100000_q{quarter}_sample1000.csv" for quarter in QUARTERS}

MODEL = "gpt-4.1-mini"
INCLUDE_AUCTION_ESTIMATES = True
INCLUDE_MARKET_INDICATORS = True
VARIANT = "with_estimates" if INCLUDE_AUCTION_ESTIMATES else "no_estimates"
TARGET_COL = "ah_price_usd"
ROWS_PER_QUARTER = 10  # Set to None for a full run.
MAX_ROWS = None
RANDOM_STATE = 42
RESUME = True
CHECKPOINT_EVERY = 10
MAX_WORKERS = 50
TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 300
MAX_RETRIES = 3
ENV_FILE = BASE_OPENAI / "env"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project dir: { ARTWORK_DIR }")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Data dir:    {ARTWORK_DATA_DIR}")
for quarter, path in QUARTER_FILES.items():
    print(f"{quarter}: {path}")


## Imports


In [ ]:
import pandas as pd

from openai_pricer import load_explicit_quarter_files, prepare_pricing_frame
from openai_pricer.runner import run_asset_pricer
from artwork_prompt_builder import build_prompt, build_system_prompt


## Load Quarter Data


In [ ]:
raw_df = load_explicit_quarter_files(QUARTER_FILES, verbose=True)
pricing_df = prepare_pricing_frame(
    raw_df,
    rows_per_quarter=ROWS_PER_QUARTER,
    max_rows=MAX_ROWS,
    random_state=RANDOM_STATE,
    dedupe_key_col="lot_url",
    verbose=True,
)
print("Rows selected for pricing:", len(pricing_df))
print(pricing_df["source_quarter"].value_counts().sort_index())
pricing_df[[c for c in ['ID', 'source_quarter', 'sale_date', 'artist_name', 'artwork_title', 'ah_low_estimate_usd', 'ah_high_estimate_usd', 'ah_price_usd'] if c in pricing_df.columns]].head()


## Preview Prompt

The prompt excludes realized sale prices and existing model prediction columns.


In [ ]:
system_prompt = build_system_prompt(INCLUDE_AUCTION_ESTIMATES)
example_prompt = build_prompt(
    pricing_df.iloc[0].to_dict(),
    INCLUDE_AUCTION_ESTIMATES,
    INCLUDE_MARKET_INDICATORS,
)
print("SYSTEM PROMPT:")
print(system_prompt[:2000])
print("\nUSER PROMPT:")
print(example_prompt[:4000])


## Run OpenAI Pricing

Requires `OPENAI_API_KEY` in `Base_OpenAI/env` or the environment. Keep `ROWS_PER_QUARTER=10` for a smoke test; set it to `None` for a full run.


In [ ]:
run_asset_pricer(
    asset_name="artwork",
    quarter_files=QUARTER_FILES,
    output_dir=OUTPUT_DIR,
    output_prefix=f"openai_gpt41mini_artwork_{VARIANT}",
    model=MODEL,
    model_label=f"OpenAI GPT-4.1 Mini Artwork - {'With Estimates' if INCLUDE_AUCTION_ESTIMATES else 'No Estimates'}",
    build_prompt=build_prompt,
    build_system_prompt=build_system_prompt,
    actual_col=TARGET_COL,
    include_auction_estimates=INCLUDE_AUCTION_ESTIMATES,
    include_market_indicators=INCLUDE_MARKET_INDICATORS,
    max_rows=MAX_ROWS,
    rows_per_quarter=ROWS_PER_QUARTER,
    random_state=RANDOM_STATE,
    resume=RESUME,
    checkpoint_every=CHECKPOINT_EVERY,
    max_workers=MAX_WORKERS,
    temperature=TEMPERATURE,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    max_retries=MAX_RETRIES,
    env_file=ENV_FILE,
)
